# Nail Retouch SD1.5 LoRA v1 - Strict Keep

This notebook prepares your processed manicure dataset and trains a first-pass **Stable Diffusion 1.5 LoRA without masks**.

Important: this v1 notebook trains on the **retouched target images plus structured captions** from `metadata.jsonl`. It is the fastest Colab-friendly baseline, and is meant to be used later with **img2img inference** for photo retouching.

## 1. Upload project

Clone the repository to Colab, then mount Google Drive and copy `processed_strict_keep` into `/content/nail-retouch-assistant/dataset`.

Expected files:
- `dataset/processed_strict_keep/train/metadata.jsonl`
- `dataset/processed_strict_keep/val/metadata.jsonl`
- `colab/training_config.yaml`
- `colab/requirements.txt`

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
import subprocess

drive.mount('/content/drive')

PROJECT_ROOT = '/content/nail-retouch-assistant'
CONFIG_PATH = f'{PROJECT_ROOT}/colab/training_config.yaml'
REQUIREMENTS_PATH = f'{PROJECT_ROOT}/colab/requirements.txt'
DRIVE_DATASET_DIR = Path('/content/drive/MyDrive/processed_strict_keep')
LOCAL_DATASET_DIR = Path(f'{PROJECT_ROOT}/dataset/processed_strict_keep')

if not DRIVE_DATASET_DIR.exists():
    raise FileNotFoundError(f'Missing dataset in Drive: {DRIVE_DATASET_DIR}')

shutil.rmtree(LOCAL_DATASET_DIR, ignore_errors=True)
LOCAL_DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(DRIVE_DATASET_DIR, LOCAL_DATASET_DIR)
print(f'Copied dataset to {LOCAL_DATASET_DIR}')

subprocess.run(f"python -m pip install -q -r {REQUIREMENTS_PATH}", shell=True, check=True)
if not Path('/content/kohya_ss').exists():
    subprocess.run(
        "git clone --recurse-submodules https://github.com/bmaltais/kohya_ss.git /content/kohya_ss",
        shell=True,
        check=True,
    )
else:
    subprocess.run(
        "cd /content/kohya_ss && git submodule update --init --recursive",
        shell=True,
        check=True,
        executable='/bin/bash',
    )

req = Path('/content/kohya_ss/requirements.txt')
lines = []
for line in req.read_text(encoding='utf-8').splitlines():
    if line.strip().startswith('-e ./sd-scripts'):
        continue
    lines.append(line)
Path('/content/kohya_ss/requirements.colab.txt').write_text('\n'.join(lines) + '\n', encoding='utf-8')

subprocess.run("python -m pip install -q -r /content/kohya_ss/requirements.colab.txt", shell=True, check=True)
subprocess.run("python -m pip install -q xformers", shell=True, check=True)
print('setup complete')

In [ ]:
from pathlib import Path
import json
import yaml
import torch

with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

project_root = Path(cfg['project_root'])
processed_dir = project_root / cfg['processed_dir']
prepared_dir = Path(cfg['prepared_data_dir'])
output_dir = Path(cfg['output_dir'])
logging_dir = Path(cfg['logging_dir'])

prepared_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
logging_dir.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError('GPU is not available. Switch Colab runtime to T4/L4/A100 before training.')
print('GPU:', torch.cuda.get_device_name(0))

cfg

In [ ]:
from pathlib import Path
import json
import shutil

def load_jsonl(path: Path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

train_rows = load_jsonl(processed_dir / cfg['train_metadata'])
shutil.rmtree(prepared_dir, ignore_errors=True)
prepared_dir.mkdir(parents=True, exist_ok=True)
instance_dir = prepared_dir / '10_manicure'
instance_dir.mkdir(parents=True, exist_ok=True)

def export_training_example(row):
    target_src = processed_dir / 'train' / row['target']
    image_name = f"{row['id']}.png"
    image_dst = instance_dir / image_name
    txt_dst = instance_dir / f"{row['id']}.txt"
    shutil.copy2(target_src, image_dst)
    txt_dst.write_text(row['prompt'], encoding='utf-8')

for row in train_rows:
    export_training_example(row)

val_prompt_file = output_dir / 'sample_prompts.txt'
val_prompt_file.write_text('\n'.join(cfg.get('sample_prompts', [])), encoding='utf-8')

print('train examples:', len(train_rows))
print('prepared dir:', instance_dir)
print('sample prompt file:', val_prompt_file)
!find /content/workdir/train_data -maxdepth 2 -type f | head -20

In [ ]:
import textwrap

train_cfg = cfg['training']
cmd = f"""
cd {cfg['kohya_repo']} && \
accelerate launch --num_cpu_threads_per_process 1 ./sd-scripts/train_network.py \
  --pretrained_model_name_or_path={cfg['base_model']} \
  --train_data_dir={prepared_dir} \
  --output_dir={output_dir} \
  --logging_dir={logging_dir} \
  --output_name={cfg['output_name']} \
  --resolution={train_cfg['resolution']} \
  --save_model_as=safetensors \
  --network_module=networks.lora \
  --network_dim={train_cfg['network_dim']} \
  --network_alpha={train_cfg['network_alpha']} \
  --train_batch_size={train_cfg['train_batch_size']} \
  --gradient_accumulation_steps={train_cfg['gradient_accumulation_steps']} \
  --max_train_epochs={train_cfg['max_train_epochs']} \
  --learning_rate={train_cfg['learning_rate']} \
  --unet_lr={train_cfg['unet_lr']} \
  --text_encoder_lr={train_cfg['text_encoder_lr']} \
  --optimizer_type={train_cfg['optimizer_type']} \
  --lr_scheduler={train_cfg['lr_scheduler']} \
  --save_every_n_epochs={train_cfg['save_every_n_epochs']} \
  --mixed_precision={train_cfg['mixed_precision']} \
  --caption_extension=.txt \
  --cache_latents \
  --cache_latents_to_disk \
  --max_data_loader_n_workers={train_cfg['max_data_loader_n_workers']} \
  --bucket_reso_steps={train_cfg['bucket_reso_steps']} \
  --min_bucket_reso={train_cfg['min_bucket_reso']} \
  --max_bucket_reso={train_cfg['max_bucket_reso']} \
  --enable_bucket \
  --xformers
"""

cmd = textwrap.dedent(cmd).strip()
print(cmd)

In [ ]:
import subprocess

process = subprocess.Popen(
    cmd,
    shell=True,
    executable='/bin/bash',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end='')

process.wait()
print('\nRETURN CODE:', process.returncode)
if process.returncode != 0:
    raise RuntimeError(f'Training failed with exit code {process.returncode}')

## 2. Expected outputs

Training should write files such as:
- `/content/workdir/outputs/nail_retouch_sd15_v1-000004.safetensors`
- `/content/workdir/outputs/nail_retouch_sd15_v1-000008.safetensors`

Use those LoRA weights later with **SD1.5 img2img** at low denoise strength for manicure photo retouching.